In [1]:
!git clone https://github.com/Giovanni24P/activation-steering.git
%cd activation-steering
!pip install -r requirements.txt

Cloning into 'activation-steering'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 13 (delta 2), reused 12 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 57.13 KiB | 5.19 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/activation-steering


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [4]:
print(len(model.model.layers))
print(model.config.hidden_size)
print(model.model.layers[18])

36
2048
Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
    (k_proj): Linear(in_features=2048, out_features=256, bias=True)
    (v_proj): Linear(in_features=2048, out_features=256, bias=True)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
    (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
    (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
)


In [5]:
def get_last_token_activation(question, instruction, layer_idx):

    activation = {}

    def hook_fn(module, inputs, output):
        if isinstance(output, tuple):
            activation["value"] = output[0]
        else:
            activation["value"] = output

    layer = model.model.layers[layer_idx]

    handle = layer.register_forward_hook(hook_fn)

    messages = [
        {
            "role": "system",
            "content": instruction
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        _ = model(**inputs)

    handle.remove()

    hidden_states = activation["value"]

    last_token = hidden_states[0, -1]

    return last_token.detach().float().cpu()

In [6]:
act = get_last_token_activation(
    "Why is the sky blue?",
    "Give a detailed and thorough explanation.",
    layer_idx=18
)

print(act.shape)

torch.Size([2048])


In [7]:
question = "Why is the sky blue?"

act_concise = get_last_token_activation(
    question,
    "Answer concisely in 1–2 sentences.",
    layer_idx=18
)

act_detailed = get_last_token_activation(
    question,
    "Give a detailed and thorough explanation.",
    layer_idx=18
)

direction = act_detailed - act_concise
print(direction.shape)

torch.Size([2048])


In [9]:
def get_contrastive_activations(question, layer_idx):

    concise = get_last_token_activation(
        question,
        "Answer concisely in 1–2 sentences.",
        layer_idx=layer_idx
    )

    detailed = get_last_token_activation(
        question,
        "Give a detailed and thorough explanation.",
        layer_idx=layer_idx
    )

    return concise, detailed


concise, detailed = get_contrastive_activations(
    "Why is the sky blue?",
    layer_idx=18
)

print(concise.shape)
print(detailed.shape)

torch.Size([2048])
torch.Size([2048])


In [11]:
import json

with open("data/prompts.json", "r") as f:
    prompts = json.load(f)

print(len(prompts))

30


In [16]:
training_prompts, testing_prompts = (prompts[:20], prompts[20:])

len(training_prompts), len(testing_prompts)

(20, 10)

In [17]:
concise_activations = []
detailed_activations = []

for item in training_prompts:

    concise, detailed = get_contrastive_activations(
        item["question"],
        layer_idx=18
    )

    concise_activations.append(concise)
    detailed_activations.append(detailed)

# convert lists into torch tensor:
concise_activations = torch.stack(concise_activations)
detailed_activations = torch.stack(detailed_activations)

print(concise_activations.shape)
print(detailed_activations.shape)

torch.Size([20, 2048])
torch.Size([20, 2048])


In [19]:
# Steering direction calculation:

mean_concise = concise_activations.mean(dim=0)
mean_detailed = detailed_activations.mean(dim=0)

steering_direction = mean_detailed - mean_concise
print(steering_direction.shape)
print(torch.norm(steering_direction))
steering_direction

torch.Size([2048])
tensor(29.5403)


tensor([-0.9616,  0.3899,  0.6371,  ...,  0.3207, -0.7458, -1.4056])

In [22]:
steering_direction_normalized = (
    steering_direction /
    torch.norm(steering_direction)
)
torch.norm(steering_direction_normalized)

tensor(1.0000)

In [23]:
v = steering_direction_normalized

concise_projection = concise_activations @ v
detailed_projection = detailed_activations @ v

print(
    concise_projection.mean().item()
)

print(
    detailed_projection.mean().item()
)

-17.794090270996094
11.746228218078613


In [25]:
import os

os.makedirs("data/activations", exist_ok=True)

In [26]:
torch.save(
    concise_activations,
    "data/activations/layer18_concise.pt"
)

torch.save(
    detailed_activations,
    "data/activations/layer18_detailed.pt"
)

torch.save(
    steering_direction,
    "data/activations/layer18_direction.pt"
)

In [31]:
torch.save(
    steering_direction_normalized,
    "data/activations/layer18_direction_normalized.pt"
)

In [36]:
layers = [8, 28]

data = []

for layer_idx in layers:
    concise_activations = []
    detailed_activations = []

    for item in training_prompts:
        concise, detailed = get_contrastive_activations(
            item["question"],
            layer_idx=layer_idx
        )

        concise_activations.append(concise)
        detailed_activations.append(detailed)

    concise_activations = torch.stack(concise_activations)
    detailed_activations = torch.stack(detailed_activations)

    mean_concise = concise_activations.mean(dim=0)
    mean_detailed = detailed_activations.mean(dim=0)

    steering_direction = mean_detailed - mean_concise

    steering_direction_normalized = (
        steering_direction /
        torch.norm(steering_direction)
    )

    print(
        f"Layer {layer_idx}: "
        f"direction norm = {torch.norm(steering_direction):.4f}"
    )

    data.append({
        "layer": f"layer{layer_idx}",
        "concise": concise_activations,
        "detailed": detailed_activations,
        "direction": steering_direction,
        "direction_normalized": steering_direction_normalized
    })

Layer 8: direction norm = 9.7582
Layer 28: direction norm = 57.2281


In [37]:
for item_layer in data:
    layer = item_layer["layer"]

    concise = item_layer["concise"]
    detailed = item_layer["detailed"]
    direction = item_layer["direction"]
    direction_normalized = item_layer["direction_normalized"]

    torch.save(
        concise,
        f"data/activations/{layer}_concise.pt"
    )

    torch.save(
        detailed,
        f"data/activations/{layer}_detailed.pt"
    )

    torch.save(
        direction,
        f"data/activations/{layer}_direction.pt"
    )

    torch.save(
        direction_normalized,
        f"data/activations/{layer}_direction_normalized.pt"
    )
